<a href="https://colab.research.google.com/github/pariupadhyay15/code-reviewer/blob/main/code_reviewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers datasets peft bitsandbytes accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.7 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/code-reviewer-llm'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
print('Project dir ready:', PROJECT_DIR)

Mounted at /content/drive
Project dir ready: /content/drive/MyDrive/code-reviewer-llm


In [3]:
from datasets import load_dataset

raw = load_dataset("ronantakizawa/github-codereview")
print(raw)
print()
print("Columns:", raw['train'].column_names)
print()
for i in [0, 1, 2]:
    ex = raw['train'][i]
    print(f"--- Example {i} ---")
    for k, v in ex.items():
        preview = str(v)[:200]
        print(f"{k}: {preview}")
    print()

README.md:   0%|          | 0.00/5.42k [00:00<?, ?B/s]

data/train/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B / 92.9MB            

data/train/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

data/train/train-00000-of-00004.parquet: reconstructing file:   0%|          |  0.00B / 94.9MB            

data/train/train-00000-of-00004.parquet: downloading bytes:           |  0.00B            

...

data/test/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/334323 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10471 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11013 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 'after_lines', 'is_negative', 'pr_title', 'pr_number', 'repo_name', 'repo_stars', 'repo_language', 'reviewer_username', 'author_username'],
        num_rows: 334323
    })
    validation: Dataset({
        features: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 'after_lines', 'is_negative', 'pr_title', 'pr_number', 'repo_name', 'repo_stars', 'repo_language', 'reviewer_username', 'author_username'],
        num_rows: 10471
    })
    test: Dataset({
        features: ['before_code', 'reviewer_comment', 'after_code', 'diff_context', 'file_path', 'comment_line', 'language', 'quality_score', 'comment_type', 'comment_length', 'before_lines', 

In [4]:
import pandas as pd

TARGET_SIZE = 35000

df = raw['train'].to_pandas()

LANG_COL = 'language'

NEG_COL = 'is_negative'

print("Language distribution (top 15):")
print(df[LANG_COL].value_counts().head(15))
print()
print("Negative (silent) vs. positive (commented) ratio:")
print(df[NEG_COL].value_counts(normalize=True))


Language distribution (top 15):
language
Python        82288
TypeScript    44309
Go            40091
Rust          38415
C++           33218
JavaScript    22347
C#            12523
C/C++         10741
Java           9579
C              6851
Kotlin         6150
Swift          5116
Vue            3423
PHP            3420
Shell          2864
Name: count, dtype: int64

Negative (silent) vs. positive (commented) ratio:
is_negative
False    0.782064
True     0.217936
Name: proportion, dtype: float64


In [5]:
frac = TARGET_SIZE / len(df)

sampled = (
    df.groupby([LANG_COL, NEG_COL], group_keys=False)
      .sample(frac=frac, random_state=42)
)

print(f"Sampled {len(sampled)} rows out of {len(df)} (target was {TARGET_SIZE})")
print()
print("Sampled language distribution (top 15):")
print(sampled[LANG_COL].value_counts().head(15))
print()
print("Sampled negative/positive ratio (should match the full-dataset ratio above):")
print(sampled[NEG_COL].value_counts(normalize=True))
#what we have done is taken a fixed amount of % from each type of language and its output true n false so our dataset has diversity

Sampled 35002 rows out of 334323 (target was 35000)

Sampled language distribution (top 15):
language
Python        8615
TypeScript    4638
Go            4198
Rust          4022
C++           3478
JavaScript    2339
C#            1311
C/C++         1124
Java          1003
C              718
Kotlin         644
Swift          535
Vue            358
PHP            358
Shell          300
Name: count, dtype: int64

Sampled negative/positive ratio (should match the full-dataset ratio above):
is_negative
False    0.782012
True     0.217988
Name: proportion, dtype: float64


In [6]:
sampled.to_parquet(f'{PROJECT_DIR}/data/train_sample_35k.parquet', index=False)
print("Saved to", f'{PROJECT_DIR}/data/train_sample_35k.parquet')

Saved to /content/drive/MyDrive/code-reviewer-llm/data/train_sample_35k.parquet


In [8]:
from transformers import AutoTokenizer
MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(tokenizer.chat_template)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba C

In [12]:
def build_message(row):
   system_msg = (
        "You are an expert code reviewer. You will be shown a code change. "
        "If there is a real problem, point it out clearly and simply. "
        "If the code looks fine, say exactly: 'No issues found.'"
    )
   user_msg = (
        f"Language: {row['language']}\n"
        f"File: {row['file_path']}\n\n"
        f"Code change:\n{row['diff_context']}"
    )
   assistant_msg = row['reviewer_comment']

   return [
        {"role": "system", "content": system_msg},
        {"role": "user", "content": user_msg},
        {"role": "assistant", "content": assistant_msg},
    ]
# system msg is the rule  user msg is msg send by user and assistant msg i sthe msg we want the model to learn

In [14]:
sample_row=sampled.iloc[0]
messages=build_message(sample_row)
for m in messages:
    print(f"--- {m['role']} ---")
    print(m['content'][:300])  # just first 300 characters, so it's not too long
    print()

--- system ---
You are an expert code reviewer. You will be shown a code change. If there is a real problem, point it out clearly and simply. If the code looks fine, say exactly: 'No issues found.'

--- user ---
Language: Assembly
File: src/rp2_common/pico_standard_link/crt0.S

Code change:
@@ -8,6 +8,7 @@
 #include "hardware/regs/m0plus.h"
 #include "hardware/regs/addressmap.h"
 #include "hardware/regs/sio.h"
+#include "pico/asm_helper.S"

--- assistant ---
Is this `#include` now redundant?



In [15]:
text= tokenizer.apply_chat_template(messages,tokenize=False, add_generation_prompt=False)
print(text)
# just testing on a single row before applying on

<|im_start|>system
You are an expert code reviewer. You will be shown a code change. If there is a real problem, point it out clearly and simply. If the code looks fine, say exactly: 'No issues found.'<|im_end|>
<|im_start|>user
Language: Assembly
File: src/rp2_common/pico_standard_link/crt0.S

Code change:
@@ -8,6 +8,7 @@
 #include "hardware/regs/m0plus.h"
 #include "hardware/regs/addressmap.h"
 #include "hardware/regs/sio.h"
+#include "pico/asm_helper.S"<|im_end|>
<|im_start|>assistant
Is this `#include` now redundant?<|im_end|>



In [18]:
def row_to_text(row):
  messages=build_message(row)
  text=tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
  return text
sampled['text']= sampled.apply(row_to_text, axis=1) #adds a new col text in dataset
print(sampled['text'].str.len().describe())

count    35002.000000
mean      1656.162391
std       1397.914240
min        361.000000
25%        848.000000
50%       1229.000000
75%       2149.000000
max      36514.000000
Name: text, dtype: float64


In [20]:
sampled['approx_token']= sampled['text'].str.len()/4
MAX_LEN=2048
too_long= (sampled['approx_token']> MAX_LEN).sum()
print(f"Rows longer than {MAX_LEN} tokens (approx): {too_long}")
print(f"That is {too_long / len(sampled) * 100:.2f}% of the data")
#the maximum ken of the text col is round 36k which we will very larger number of token for the model as there is a limit of context
#length. so dropping the 0.78% data which is larger than 2048 tokens

Rows longer than 2048 tokens (approx): 272
That is 0.78% of the data


In [22]:
before_count = len(sampled)

sampled = sampled[sampled['approx_token'] <= MAX_LEN].reset_index(drop=True)
#                 this marks true and false
# this keeps only true
#                                                     this resets the index

after_count = len(sampled)
print(f"Dropped {before_count - after_count} rows")
print(f"Remaining rows: {after_count}")

Dropped 272 rows
Remaining rows: 34730


In [23]:
sampled.to_parquet(f'{PROJECT_DIR}/data/train_sample_ready.parquet', index=False)
print("Saved final training data to Drive")

Saved final training data to Drive
